In [ ]:
import pandas as pd

# Load the Excel file
df = pd.read_excel("Muscle_Male_Female_All.xlsx")

# Sanity checks
print(df.shape)
df.head()
df.info()
df.isna().sum()  # check for missing values

In [ ]:
xl = pd.ExcelFile("Muscle_Male_Female_All.xlsx")
print(xl.sheet_names)

In [ ]:
import pandas as pd

raw = pd.read_excel("Muscle_Male_Female_All.xlsx", sheet_name="Male_Female_All")

#Female block: columns 0–10
female = raw.iloc[:, 0:11].copy().set_index("Gene")

#Male block: columns 12–22
male = raw.iloc[:, 12:23].copy()
male.columns = male.columns.str.replace(".1", "", regex=False)
male = male.set_index("Gene")

print("Female:", female.shape)
print("Male:", male.shape)
female.head()

In [ ]:
female.isna().sum()

In [ ]:
def restructure(df):
    """Split the Gene-indexed (logFC, FDR, sample counts...) table into:
    1. expression(expr): samples-as-rows, genes-as-columns matrix with Sample/Group/Label
    2. stats: gene-level logFC/FDR table (kept separate)
    """
    stats = df[["logFC", "FDR"]].copy()
    counts = df.drop(columns=["logFC", "FDR"])

    n_dup = counts.index.duplicated().sum()
    if n_dup:
        print(f"Warning: {n_dup} duplicate gene symbols found in index")

    expr = counts.T # transpose to have samples as rows and genes as columns
    expr.index.name = "Sample" # rename index to Sample
    expr = expr.reset_index() # reset index so it indexes 0-7 for Sham and C26 samples.
    expr["Group"] = expr["Sample"].apply(lambda s: "SHAM" if "SHAM" in s else "C26") # assign Group based on Sample name
    expr["Label"] = (expr["Group"] == "C26").astype(int) # assign Label as 1 for C26 and 0 for SHAM

    return expr, stats


female_expr, female_stats = restructure(female)
male_expr, male_stats = restructure(male)

# Structure check
print("Female expr:", female_expr.shape, "| stats:", female_stats.shape)
print("Male expr:", male_expr.shape, "| stats:", male_stats.shape)
print("Female group counts:", female_expr["Group"].value_counts().to_dict())
print("Male group counts:", male_expr["Group"].value_counts().to_dict())
print("Female samples:", female_expr["Sample"].tolist())
print("Male samples:", male_expr["Sample"].tolist())

In [ ]:
female_expr

In [ ]:
import os

# Save restructured output as NEW files
out_dir = "restructured"
os.makedirs(out_dir, exist_ok=True)

female_expr.to_csv(os.path.join(out_dir, "female_expr.csv"), index=False)
male_expr.to_csv(os.path.join(out_dir, "male_expr.csv"), index=False)
female_stats.to_csv(os.path.join(out_dir, "female_stats.csv"))
male_stats.to_csv(os.path.join(out_dir, "male_stats.csv"))

print(f"Saved restructured files to ./{out_dir}/ (source .xlsx untouched).")

In [ ]:
SIG_THRESHOLD = 0.05  # standard FDR cutoff

female_sig_genes = female_stats.index[female_stats["FDR"] < SIG_THRESHOLD]
male_sig_genes = male_stats.index[male_stats["FDR"] < SIG_THRESHOLD]

female_expr_sig = female_expr[["Sample", "Group", "Label"] + list(female_sig_genes)]
male_expr_sig = male_expr[["Sample", "Group", "Label"] + list(male_sig_genes)]

print(f"Female: {len(female_sig_genes)} / {len(female_stats)} genes significant (FDR < {SIG_THRESHOLD})")
print(f"Male: {len(male_sig_genes)} / {len(male_stats)} genes significant (FDR < {SIG_THRESHOLD})")

In [ ]:
female_expr_sig

In [ ]:
EXPR_THRESHOLD = 50 #filter out genes with low expression (mean counts < EXPR_THRESHOLD) in either group

def filter_low_expression(expr_df, sig_genes, threshold = EXPR_THRESHOLD):
    group_means = expr_df.groupby("Group")[sig_genes].mean()
    keep_genes = []
    for gene in sig_genes: 
        if (group_means.loc["SHAM", gene] >= threshold) or (group_means.loc["C26", gene] >= threshold):
            keep_genes.append(gene)
    return keep_genes

female_gene_cols = [c for c in female_expr_sig.columns if c not in ("Sample", "Group", "Label")]
male_gene_cols = [c for c in male_expr_sig.columns if c not in ("Sample", "Group", "Label")]

female_keep_genes = filter_low_expression(female_expr_sig, female_gene_cols)
male_keep_genes = filter_low_expression(male_expr_sig, male_gene_cols)

female_expr_filtered = female_expr_sig[["Sample", "Group", "Label"] + list(female_keep_genes)]
male_expr_filtered = male_expr_sig[["Sample", "Group", "Label"] + list(male_keep_genes)]

print(f"Female: {len(female_keep_genes)} / {len(female_gene_cols)} genes pass expression filter (mean >= {EXPR_THRESHOLD} in at least one group)")
print(f"Male: {len(male_keep_genes)} / {len(male_gene_cols)} genes pass expression filter (mean >= {EXPR_THRESHOLD} in at least one group)")


In [ ]:
female_expr_filtered

In [ ]:
with open(os.path.join(out_dir, "female_filtered_gene_symbols.txt"), "w") as f:
    f.write("\n".join(female_keep_genes))

with open(os.path.join(out_dir, "male_filtered_gene_symbols.txt"), "w") as f:
    f.write("\n".join(male_keep_genes))

print(f"Saved {len(female_keep_genes)} female / {len(male_keep_genes)} male filtered gene symbols to ./{out_dir}/")